In [7]:
import os
import sys
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [2]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气信息字符串
    """
    # 模拟天气数据（实际应用中应调用真实API）
    weather_data = {
        "北京": "晴天，温度 15°C，空气质量良好",
        "上海": "多云，温度 18°C，有轻微雾霾",
        "深圳": "阴天，温度 22°C，可能有小雨",
        "成都": "小雨，温度 12°C，湿度较高"
    }

    return weather_data.get(city, f"抱歉，暂时没有{city}的天气数据")



In [10]:
print("测试天气工具：")
print(f"北京天气: {get_weather.invoke({'city': '北京'})}")
print(f"上海天气: {get_weather.invoke({'city': '上海'})}")
print(f"河南: {get_weather.invoke({'city': '河南'})}")
print("-----------")
print("get_weather.name",get_weather.name)
print("get_weather.description",get_weather.description)
print("get_weather.args",get_weather.args)

测试天气工具：
北京天气: 晴天，温度 15°C，空气质量良好
上海天气: 多云，温度 18°C，有轻微雾霾
河南: 抱歉，暂时没有河南的天气数据
-----------
get_weather.name get_weather
get_weather.description 获取指定城市的天气信息

参数:
    city: 城市名称，如"北京"、"上海"

返回:
    天气信息字符串
get_weather.args {'city': {'title': 'City', 'type': 'string'}}


In [ ]:
@tool
def get_current_time() -> str:
    """获取当前时间"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


print("工具名称:", get_current_time.name)
print("工具描述:", get_current_time.description)
print("工具参数:", get_current_time.args)

# 调用工具
result = get_current_time.invoke({})
# 被 @tool 装饰器装饰的函数会被转换为 LangChain 的 Tool
# 对象，这个对象有 .invoke() 方法。
print(f"\n调用结果: {result}")

print("\n💡 关键点：")
print("  1. @tool 装饰器会自动提取函数名、docstring、参数")
print("  2. docstring 很重要！AI 用它理解工具的功能")
print("  3. 类型注解帮助 AI 理解参数类型")


工具名称: get_current_time
工具描述: 获取当前时间
工具参数: {}

调用结果: 2026-03-31 11:48:31

💡 关键点：
  1. @tool 装饰器会自动提取函数名、docstring、参数
  2. docstring 很重要！AI 用它理解工具的功能
  3. 类型注解帮助 AI 理解参数类型


In [12]:
print(f"名称: {get_weather.name}")
print("-----")
print(f"描述: {get_weather.description}")
print("-----")
print(f"参数: {get_weather.args}")

名称: get_weather
-----
描述: 获取指定城市的天气信息

参数:
    city: 城市名称，如"北京"、"上海"

返回:
    天气信息字符串
-----
参数: {'city': {'title': 'City', 'type': 'string'}}


In [14]:
@tool
def calculator(operation: str, a: float, b: float) -> str:
    """
    执行基本的数学计算

    参数:
        operation: 运算类型，支持 "add"(加), "subtract"(减), "multiply"(乘), "divide"(除)
        a: 第一个数字
        b: 第二个数字

    返回:
        计算结果字符串
    """
    operations = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y != 0 else "错误：除数不能为零"
    }

    if operation not in operations:
        return f"不支持的运算类型：{operation}。支持的类型：add, subtract, multiply, divide"

    try:
        result = operations[operation](a, b)
        return f"{a} {operation} {b} = {result}"
    except Exception as e:
        return f"计算错误：{e}"

In [15]:
print(f"名称: {calculator.name}")
print("-----")
print(f"描述: {calculator.description}")
print("-----")
print(f"参数: {calculator.args}")

名称: calculator
-----
描述: 执行基本的数学计算

参数:
    operation: 运算类型，支持 "add"(加), "subtract"(减), "multiply"(乘), "divide"(除)
    a: 第一个数字
    b: 第二个数字

返回:
    计算结果字符串
-----
参数: {'operation': {'title': 'Operation', 'type': 'string'}, 'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


In [22]:
model_with_tools = model.bind_tools([get_weather, calculator])
print(model_with_tools)

bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x123f6ac10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1243432d0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********')) kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': '获取指定城市的天气信息\n\n参数:\n    city: 城市名称，如"北京"、"上海"\n\n返回:\n    天气信息字符串', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'calculator', 'description': '执行基本的数学计算\n\n参数:\n    operation: 运算类型，支持 "add"(加), "subtract"(减), "multiply"(乘), "divide"(除)\n    a: 第一个数字\n    b: 第二个数字\n\n返回:\n    计算结果字符串

In [33]:
print(type(model_with_tools))
print(model_with_tools.profile)
print(model_with_tools.profile.get("max_input_tokens","不存在"))
print(model_with_tools.profile.get("max_tokens","不存在"))
print(model_with_tools.profile["max_input_tokens"])
print(model_with_tools.client)
print(model_with_tools.kwargs)
print(model_with_tools.kwargs.get("tools"))
print(model_with_tools.kwargs.get("tools")[0])
print(model_with_tools.kwargs.get("tools")[1])
print(model_with_tools.kwargs.get("tools")[1].get("type"))
print(model_with_tools.kwargs.get("tools")[1].get("function"))
print(model_with_tools.kwargs.get("tools")[1].get("function").get("name"))
print(model_with_tools.kwargs.get("tools")[1].get("function").get("description"))
print(model_with_tools.kwargs.get("tools")[1].get("function").get("parameters"))

<class 'langchain_core.runnables.base.RunnableBinding'>
{'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}
131072
不存在
131072
{'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': '获取指定城市的天气信息\n\n参数:\n    city: 城市名称，如"北京"、"上海"\n\n返回:\n    天气信息字符串', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'calculator', 'description': '执行基本的数学计算\n\n参数:\n    operation: 运算类型，支持 "add"(加), "subtract"(减), "multiply"(乘), "divide"(除)\n    a: 第一个数字\n    b: 第二个数字\n\n返回:\n    计算结果字符串', 'parameters': {'properties': {'operation': {'type': 'string'}, 'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['operation', 'a', 'b'], 'type': 'object'}}}]}
[{'type': 'function', 'function': {'n

In [39]:
response = model_with_tools.invoke("北京今天天气怎么样？")
print(response)
print("-------------")
print(response.additional_kwargs)
print(response.tool_calls)

content='' additional_kwargs={'tool_calls': [{'id': 'swx340360', 'function': {'arguments': '{"city":"北京"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 452, 'total_tokens': 466, 'completion_time': 0.038546921, 'completion_tokens_details': None, 'prompt_time': 0.052246229, 'prompt_tokens_details': None, 'queue_time': 0.110581594, 'total_time': 0.09079315}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d43bf-834f-7e73-add9-9f1e2b60095a-0' tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'swx340360', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 452, 'output_tokens': 14, 'total_tokens': 466}
-------------
{'tool_calls': [{'id': 'swx340360', 'function': {'arguments': '{"city":"北京"}', 'name': 'get_weather'}, 'type': 

In [40]:
if response.tool_calls:
    print(f"AI 决定使用工具！")
    print(f"工具调用: {response.tool_calls}")
else:
    print(f"AI 直接回答（未使用工具）")
    print(f"回复: {response.content}")

AI 决定使用工具！
工具调用: [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'swx340360', 'type': 'tool_call'}]
